# LAB 08 - Tokenization, special tokens, attention mask, MLM

Tokenizer `bert-base-uncased` (WordPiece, cùng vocabulary với BERT-mini `google/bert_uncased_L-4_H-256_A-4`). Model và tokenizer load offline từ `../hf_models`.

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("../lab4"))
from pretrained_classification import MODEL_NAME, load_tokenizer     # đặt HF_HOME = ../hf_models

import numpy as np
import pandas as pd
import torch
from transformers import AutoModelForMaskedLM
from preprocessing import clean_text, deduplicate, load_imdb, preprocess, split_train_val

tok = load_tokenizer()
print("vocab size:", tok.vocab_size)
pd.DataFrame({"token": tok.all_special_tokens, "id": tok.all_special_ids})

vocab size: 30522


,token,id
0,[UNK],100
1,[SEP],102
2,[PAD],0
3,[CLS],101
4,[MASK],103


## 1. "The cats are playing."

In [2]:
text = "The cats are playing."
enc = tok(text)
print("keys:", list(enc.keys()))
pd.DataFrame({"token": tok.convert_ids_to_tokens(enc["input_ids"]), "input_id": enc["input_ids"],
              "attention_mask": enc["attention_mask"], "token_type_id": enc["token_type_ids"]})

keys: ['input_ids', 'token_type_ids', 'attention_mask']


,token,input_id,attention_mask,token_type_id
0,[CLS],101,1,0
1,the,1996,1,0
2,cats,8870,1,0
3,are,2024,1,0
4,playing,2652,1,0
5,.,1012,1,0
6,[SEP],102,1,0


- `input_ids`: chỉ số của từng token trong vocabulary, là input của embedding layer.
- `attention_mask`: 1 là token thật, 0 là padding. Model không attend tới vị trí có mask 0.
- `token_type_ids`: đánh dấu câu A (0) hay câu B (1) khi đưa một cặp câu, ví dụ `[CLS] A [SEP] B [SEP]`.

Câu có 4 từ, nhưng thành 7 token: 4 từ, dấu ".", cùng `[CLS]` và `[SEP]`. Không từ nào bị tách subword.

## 2. Words, tokens, subwords

In [3]:
examples = ["The cats are playing.", "This movie was unwatchable!", "Tokenization of Schwarzenegger's films",
            "It was sooooo baaaad, 2/10.", "I absolutely loved the cinematography."]
rows = []
for s in examples:
    toks = tok.tokenize(s)
    rows.append({"text": s, "words": len(s.split()), "tokens": len(toks),
                 "subword tokens (##)": sum(t.startswith("##") for t in toks), "tokenization": " ".join(toks)})
pd.DataFrame(rows)

,text,words,tokens,subword tokens (##),tokenization
0,The cats are playing.,4,5,0,the cats are playing .
1,This movie was unwatchable!,4,7,2,this movie was un ##watch ##able !
2,Tokenization of Schwarzenegger's films,4,9,3,token ##ization of schwarz ##ene ##gger ' s films
3,"It was sooooo baaaad, 2/10.",5,13,4,"it was soo ##oo ##o ba ##aa ##ad , 2 / 10 ."
4,I absolutely loved the cinematography.,5,6,0,i absolutely loved the cinematography .


Từ hiếm, tên riêng và từ viết sai ("baaaad") bị tách thành nhiều subword thay vì thành `[UNK]`.

## 3. Padding và attention mask

In [4]:
batch = tok(["the cat eats fish", "the cat eats fish every single day"], padding=True, truncation=True, return_tensors="pt")
for ids, m in zip(batch["input_ids"], batch["attention_mask"]):
    print(tok.convert_ids_to_tokens(ids), m.tolist())

['[CLS]', 'the', 'cat', 'eats', 'fish', '[SEP]', '[PAD]', '[PAD]', '[PAD]'] [1, 1, 1, 1, 1, 1, 0, 0, 0]
['[CLS]', 'the', 'cat', 'eats', 'fish', 'every', 'single', 'day', '[SEP]'] [1, 1, 1, 1, 1, 1, 1, 1, 1]


## 4. Tokenization trên IMDb

So sánh với vocabulary theo từ (20,000 từ) của LAB 05 - 07.

In [5]:
train_raw, test_raw = load_imdb()
train_df, _ = split_train_val(deduplicate(train_raw), val_size=0.2, seed=42)
texts = [clean_text(t) for t in train_df["text"]]
lengths = np.array([len(x) for x in tok(texts, add_special_tokens=True)["input_ids"]])
words = [preprocess(t) for t in train_df["text"]]
n_words = np.array([len(w) for w in words])
wp = [tok.tokenize(t) for t in texts[:2000]]
stats = {
    "median WordPiece tokens / review": float(np.median(lengths)),
    "median words / review": float(np.median(n_words)),
    "tokens / word": float(lengths.sum() / n_words.sum()),
    "% token là subword (##)": 100 * float(np.mean([t.startswith("##") for x in wp for t in x])),
    "% token là [UNK]": 100 * float(np.mean([t == "[UNK]" for x in wp for t in x])),
}
for L in [64, 128, 256, 512]:
    stats[f"% review không bị cắt ở {L}"] = 100 * float((lengths <= L).mean())
    stats[f"% token giữ lại ở {L}"] = 100 * float(np.minimum(lengths, L).sum() / lengths.sum())
pd.Series(stats).round(2)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (614 > 512). Running this sequence through the model will result in indexing errors


median WordPiece tokens / review    221.00
median words / review               174.00
tokens / word                         1.27
% token là subword (##)               5.69
% token là [UNK]                      0.00
% review không bị cắt ở 64            2.12
% token giữ lại ở 64                 21.45
% review không bị cắt ở 128          12.19
% token giữ lại ở 128                41.50
% review không bị cắt ở 256          58.80
% token giữ lại ở 256                68.34
% review không bị cắt ở 512          86.76
% token giữ lại ở 512                89.12
dtype: float64

WordPiece gần như không có `[UNK]`, trong khi vocabulary 20,000 từ của LAB 05 có 2.6% token là UNK. Đổi lại, một review dài hơn khi tính bằng token, nên với cùng max_length thì giữ được ít nội dung hơn.

## 5. Masked Language Modeling với BERT-mini

In [6]:
mlm = AutoModelForMaskedLM.from_pretrained(MODEL_NAME).eval()

def mask_predict(sentence, answer, k=5):
    enc = tok(sentence, return_tensors="pt")
    pos = (enc["input_ids"][0] == tok.mask_token_id).nonzero()[0, 0]
    with torch.no_grad():
        p = mlm(**enc).logits[0, pos].softmax(-1)
    top = p.topk(k)
    p_ans = float(p[tok.convert_tokens_to_ids(answer)])
    return {"sentence": sentence, "answer": answer, "P(answer)": p_ans, "MLM loss = -ln P": -np.log(p_ans),
            "top-5": ", ".join(f"{tok.convert_ids_to_tokens(int(j))} {float(v):.3f}" for v, j in zip(top.values, top.indices))}

pd.DataFrame([mask_predict("The cat [MASK] fish.", "eats"),
              mask_predict("Paris is the capital of [MASK].", "france"),
              mask_predict("This movie was [MASK], I loved every minute.", "great"),
              mask_predict("The acting was terrible and the plot was [MASK].", "boring")]).round(3)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: google/bert_uncased_L-4_H-256_A-4
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,sentence,answer,P(answer),MLM loss = -ln P,top-5
0,The cat [MASK] fish.,eats,0.098,2.320,"eats 0.098, is 0.093, are 0.035, ##a 0.025, ha..."
1,Paris is the capital of [MASK].,france,0.048,3.026,"paris 0.688, france 0.048, brussels 0.022, bor..."
2,"This movie was [MASK], I loved every minute.",great,0.010,4.566,"amazing 0.041, beautiful 0.035, real 0.031, in..."
3,The acting was terrible and the plot was [MASK].,boring,0.001,6.682,"terrible 0.029, impossible 0.025, complete 0.0..."


BERT-mini chưa fine-tune đã có kiến thức từ pretraining: đoán đúng "eats" đứng đầu, và với câu khen/chê thì top-5 là từ đúng cực tính ("amazing", "beautiful" / "terrible", "impossible"). Nhưng model nhỏ còn sai rõ: "Paris is the capital of [MASK]" lại đoán "paris" (0.688). Loss $-\ln P$ nhỏ khi model tự tin vào đúng token và tăng nhanh khi model bỏ qua token đúng ("boring": 6.68).